# 50.007 Machine Learning - GenAI Content Detection

**Binary text classification: `1` = machine-generated, `0` = human-authored. Scored on Macro F1.**

This is the single submission notebook. It contains Tasks 1, 2 and 3, each in its own
clearly labelled part.

| Part | Task | Deliverable |
|---|---|---|
| Part 1 | Logistic regression implemented from scratch | `submissions/LogReg_predictions.csv` and `submissions/LogReg_Prediction.csv` |
| Part 2 | PCA + KNN (`n_neighbors=2`) at 2000 / 1000 / 500 / 100 components | four `submissions/pca_knn_*_components.csv` |
| Part 3 | Race to the top, non-deep-learning | `submissions/tuned_lgbm_pergroup62_49.csv` |

**How to run.** Every part reads only from `data/raw/` and writes to `submissions/`.
Part 3 rebuilds its features from the raw text and refits the model, which takes roughly
25 minutes end to end; Parts 1 and 2 take a few minutes each. Nothing depends on a
cached artifact.

**On the two filenames in Part 1.** The course brief slide names the Task 1 file
`LogReg_predictions.csv`; the written task sheet names it `LogReg_Prediction.csv`. We
could not establish which the grader uses, so both are written with identical contents.

## Part 0. Setup

In [ ]:
# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import numpy as np
import pandas as pd

from src import paths, data, evaluation, ensemble, text, clustering
from src import text_features as tf

RANDOM_STATE = 42

---

# PART 1 - TASK 1: Logistic Regression from scratch

**Constraint:** no `sklearn.LogisticRegression` and no other pre-built logistic
regression anywhere in the path. Everything below is NumPy. The five functions the brief
names are implemented exactly as specified: `sigmoid`, `loss`, `gradients`, `train`,
`predict`.

Input is the supplied 5,000-column TF-IDF matrix, as the brief requires for Tasks 1
and 2.

In [ ]:
X, y, ids = data.load_train_features()
dev_idx = np.load(paths.DATA_PROCESSED / "dev_idx.npy")
holdout_idx = np.load(paths.DATA_PROCESSED / "holdout_idx.npy")
print(f"train {X.shape}, machine share {y.mean():.4f}")
print(f"dev {len(dev_idx)}, holdout {len(holdout_idx)}")

### 1.1 The five required functions

In [ ]:
def sigmoid(z):
    """Numerically stable logistic sigmoid."""
    z = np.asarray(z, dtype=float)
    # Calculate negative and positive z separately to prevent floating-point errors
    result = np.empty_like(z)
    positive = z >= 0
    result[positive] = 1 / (1 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1 + exp_z)
    return result.item() if result.ndim == 0 else result


def loss(y, y_hat):
    """Binary log loss."""
    # Clip to prevent log(0) or log(1)
    y_hat = np.clip(y_hat, np.finfo(float).eps, 1 - np.finfo(float).eps)
    m = y.shape[0]
    inner = np.sum(y * np.log(y_hat) + (1-y) * np.log(1 - y_hat))
    return -inner/m


def gradients(X, y, y_hat):
    """Return (dw, db)."""
    m = y.shape[0]
    dw = np.matmul(X.T, y_hat-y)/m
    db = np.sum(y_hat - y)/m
    return (dw, db)


def train(X, y, bs, epochs, lr):
    """Mini-batch gradient descent. Returns learned (w, b) and loss history."""
    loss_history = []
    b = 0
    w = np.ones(X.shape[1])
    y_hat = sigmoid(np.matmul(w, X.T) + b)
    loss_history.append(loss(y, y_hat))
    for i in range(epochs):
        for start in range(0, X.shape[0], bs):
            X_batch = X[start:start + bs]
            y_batch = y[start:start + bs]
            y_hat = sigmoid(np.matmul(w, X_batch.T) + b)
            dw, db = gradients(X_batch, y_batch, y_hat)
            w = w - lr * dw
            b = b - lr * db
        y_hat = sigmoid(np.matmul(w, X.T) + b)
        loss_history.append(loss(y, y_hat))
    return w, b, loss_history


def predict(X, w, b, threshold=0.5):
    """Return 0/1 labels."""
    return np.where(sigmoid(np.matmul(w, X.T) + b) >= threshold, 1, 0)

### 1.2 Training

`bs`, `epochs` and `lr` were chosen by hand on the dev split. The batch size is small
because the loss surface here is well conditioned and small batches converged faster per
epoch than full-batch descent; the learning rate is the largest that did not oscillate.

In [ ]:
BS, EPOCHS, LR = 4, 300, 0.1

w, b, hist = train(X[dev_idx], y[dev_idx], bs=BS, epochs=EPOCHS, lr=LR)

f1 = evaluation.macro_f1(y[holdout_idx], predict(X[holdout_idx], w, b))
print(f"bs={BS}, epochs={EPOCHS}, lr={LR}")
print(f"holdout Macro F1: {f1:.4f}")
print(f"final training loss: {hist[-1]:.5f}, still falling by "
      f"{hist[-11] - hist[-1]:.6f} over the last 10 epochs")

**Comparison against sklearn, which the rubric asks for.** sklearn's
`LogisticRegression` on the same split and the same features reaches **0.7392** Macro F1
against our **0.7288**, a gap of 0.0104.

The gap is not an error in the gradient or the loss, which we checked against numerical
differentiation. It is that plain mini-batch gradient descent has not converged when we
stop it, as the loss history above shows, while sklearn's solver runs to a tolerance and
applies L2 regularisation by default. We report the honest number rather than tuning the
from-scratch model until it matched.

### 1.3 Predict the test set and write both filenames

In [ ]:
Xt, test_ids = data.load_test_features()
preds = predict(Xt, w, b)

# Both spellings ship: the brief slide and the task sheet disagree, and this is an
# exact-name deliverable. Identical contents, so whichever the grader reads is correct.
for fname in ["LogReg_predictions.csv", "LogReg_Prediction.csv"]:
    data.write_submission(test_ids, preds, fname)

sample = pd.read_csv(paths.DATA_RAW / "sample_submission.csv", dtype={"id": str})
for fname in ["LogReg_predictions.csv", "LogReg_Prediction.csv"]:
    sub = pd.read_csv(paths.SUBMISSIONS / fname, dtype={"id": str})
    assert list(sub.columns) == ["id", "label"]
    assert list(sub["id"]) == list(sample["id"]), "id order drifted"
    assert set(sub["label"].unique()) <= {0, 1}
    print(f"{fname}: {len(sub)} rows, predicted machine share {sub['label'].mean():.4f}")

---

# PART 2 - TASK 2: PCA and KNN at four component counts

PCA on the supplied 5,000 TF-IDF features, then KNN with `n_neighbors=2`. sklearn is
allowed here. The four component counts are fixed by the task, so this is the one place
in the project where a structural hyperparameter is not chosen from a diagnostic.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

COMPONENT_COUNTS = [2000, 1000, 500, 100]

for n in COMPONENT_COUNTS:
    pipe = make_pipeline(PCA(n_components=n, random_state=RANDOM_STATE),
                         KNeighborsClassifier(n_neighbors=2))
    pipe.fit(X, y)
    data.write_submission(test_ids, pipe.predict(Xt), f"pca_knn_{n}_components.csv")
    print(f"{n:>5} components: submission written")

### 2.1 Analysis of the reduced components

In [ ]:
# One 2000-component fit serves every point on the curve: PCA components are
# variance-ordered, so the first n columns of a 2000-fit are exactly the n-fit.
pca_full = PCA(n_components=2000, random_state=RANDOM_STATE).fit(X)
cum_evr = np.cumsum(pca_full.explained_variance_ratio_)

# Test Macro F1 read back from the Kaggle leaderboard for each of the four submissions.
KAGGLE_MACRO_F1 = {100: 0.67793, 500: 0.66373, 1000: 0.55923, 2000: 0.41495}

PRIOR = y.mean()
FLOOR = PRIOR ** 2      # share if both neighbours were independent draws from the prior

rows = []
for n in sorted(KAGGLE_MACRO_F1):
    lab = pd.read_csv(paths.SUBMISSIONS / f"pca_knn_{n}_components.csv",
                      dtype={"id": str})["label"].to_numpy()
    rows.append({"components": n, "cum_evr": cum_evr[n - 1], "share": lab.mean(),
                 # q solves share = q**2: the rate at which one retrieved neighbour is
                 # machine, given that both must be for the model to say machine
                 "implied_q": np.sqrt(lab.mean()), "kaggle_f1": KAGGLE_MACRO_F1[n]})

shares = pd.DataFrame(rows)
print(shares.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\ntraining prior {PRIOR:.4f}, independent-draw floor {FLOOR:.4f}")
print(f"90% of variance is not reached within 2000 components "
      f"(max {cum_evr[-1]:.2%})")

**Keeping more variance made the classifier monotonically worse.** The best score is the
most aggressive reduction, 100 components, holding under a sixth of the variance. Two
mechanisms compound.

**Distance concentration.** As dimension grows the nearest and farthest neighbour
distances converge, and that contrast is what KNN treats as similarity. A component
contributes to squared distance in proportion to its variance, and there are so many tail
components that at 2,000 components about 79% of the squared distance comes from
directions outside the leading 100. Euclidean distance cannot down-weight them.

**The k=2 tie-break, which is the larger effect.** With two neighbours and two classes
the vote can split 1-1; sklearn resolves ties by lowest class index, which here is human.
The model says machine only when *both* neighbours are machine, so any loss of retrieval
quality is squared. The `share` column measures it: if degraded retrieval merely made the
neighbours independent draws from the 62.52% prior, the share would fall towards 0.3909
and stop, because that is a floor under that account. It reaches 0.0590, far below the
floor, so the retrieved neighbours are systematically human rather than random. The
implied per-neighbour machine rate falls 0.738, 0.607, 0.420, 0.243 against a prior of
0.6252 it should have matched throughout.

The standard explanation for that shape is **hubness**, where a few training points
become the nearest neighbour of disproportionately many queries; if those hubs skew
human, the tie fires for most queries. We did not measure the k-occurrence distribution,
so we report this as the leading explanation rather than a demonstrated one. Either way
the practical conclusion is the same: the failure is as much the fixed `n_neighbors=2`
tie-break as the dimensionality, and `k=3` or distance weighting would break ties on
evidence instead of class index. Both are outside the task's fixed specification.

---

# PART 3 - TASK 3: Race to the top

No deep learning and no LLMs. The final model is **LightGBM on 40,385 features built
from the raw text**, with the decision threshold set separately for each of the two
populations visible in the test file.

## 3.1 Every model we tried, with its key hyperparameters

Documented here as the brief requires. Scores are Macro F1; which validation protocol
each was measured under is stated, because the project used two.

### Stage 1 - twelve baselines at library defaults, supplied features

5-fold stratified CV, `random_state=42`, no tuning, so the comparison is of model
families rather than of search effort (`data/processed/baseline_results.csv`).

| Rank | Model | CV Macro F1 | Rank | Model | CV Macro F1 |
|---|---|---|---|---|---|
| 1 | LightGBM | 0.7391 | 7 | LogReg lasso (L1) | 0.7112 |
| 2 | XGBoost | 0.7289 | 8 | RandomForest | 0.6871 |
| 3 | HistGradientBoosting | 0.7262 | 9 | ComplementNB | 0.6580 |
| 4 | LogReg ridge (L2) | 0.7246 | 10 | KNN | 0.6542 |
| 5 | LinearSVC | 0.7228 | 11 | MultinomialNB | 0.6505 |
| 6 | LogReg elastic net | 0.7196 | 12 | AdaBoost | 0.5978 |

### Stage 2 - the five carried forward and tuned, supplied features

Two-stage search each (coarse `RandomizedSearchCV`, then a refined grid around the
winner), scored on the same 5-fold protocol and confirmed on an untouched 4,000-row
holdout (`data/processed/holdout_metrics.csv`).

| Model | Key hyperparameters searched | CV | Holdout |
|---|---|---|---|
| **LightGBM** | `learning_rate`, `n_estimators`, `num_leaves`, `min_child_samples`, `colsample_bytree`, `reg_alpha`, `reg_lambda` | **0.7443** | **0.7437** |
| Linear ensemble | rank blend, hill-climbed weights, meta-GBM stacker over 6 members | 0.7307 | 0.7396 |
| LogReg elastic net | `C`, `l1_ratio`, `saga` solver | 0.7290 | 0.7395 |
| LogReg ridge | `C` log-uniform, `class_weight="balanced"` | 0.7279 | 0.7392 |
| LinearSVC | `C=0.139824`, `loss="squared_hinge"` | 0.7275 | 0.7414 |

### Stage 3 - on the raw-text features

Standard 5-fold and 3-band grouped CV on the `chosen` 40,385-column matrix
(`data/processed/chosen2_results.csv`), plus the linear family measured on uncapped
n-grams alone (`data/processed/uncapped_ngram_results.csv`, standard CV).

| Model | Key hyperparameters | Standard CV | Grouped CV |
|---|---|---|---|
| **LightGBM, tuned** | the configuration in 3.4 below | **0.8849** | **0.8135** |
| LightGBM, defaults | `class_weight="balanced"` only | 0.8764 | 0.8089 |
| XGBoost, defaults | `max_bin=64`, lowered from 256 to fit in memory | 0.8758 | 0.8069 |
| LinearSVC, scaled | `C=1` | 0.7948 | 0.7411 |
| ExtraTrees | `n_estimators=300` | 0.7306 | 0.5908 |
| LinearSVC on uncapped n-grams | `C=1`, 618,408 columns | 0.8160 | - |
| LogisticRegression on uncapped n-grams | `C=10` | 0.8158 | - |
| SGDClassifier on uncapped n-grams | `loss="modified_huber"` | 0.8149 | - |
| NBSVM on uncapped n-grams | `C=1`, NB log-count ratio interpolation | 0.8027 | - |

Fifteen distinct model families in total. Three directions were tested and rejected on
evidence, and are recorded as negatives rather than dropped: uncapping the n-gram
vocabularies (a real local gain of +0.0126 grouped CV that then lost 0.0028 on the
leaderboard), NBSVM (below plain LinearSVC everywhere we tested it), and ensembling
(+0.00171, inside the noise floor).

**ExtraTrees is worth one line as a mechanism, not just a score.** It was a strong
ensemble member on the supplied 5,000 features and collapses to 0.5908 grouped here,
because it samples `sqrt(618793)` of about 787 columns per split while the 385 dense
style columns are 0.06% of the matrix, so a typical split sees under one informative
feature. Random-subspace methods need `max_features` raised before they can be used on a
matrix this wide.

## 3.2 Features

Eight blocks built from `data/raw/train.csv` and `test.csv`. Blocks A to F are
per-document statistics that cannot leak between rows; H and I are vectorizers fitted on
training text only. `G_readability` was dropped by the ablation in notebook 14 because it
cost nothing to remove.

In [ ]:
CHOSEN = ["A_function_words", "B_punctuation", "C_casing", "D_structure",
          "E_length", "F_diversity", "H_char_ngrams", "I_word_ngrams"]

train_ids, train_texts, y_full = text.load_train_text()
test_ids_txt, test_texts = text.load_test_text()

# Builds from raw text rather than loading our cache, so this runs standalone.
# H is char_wb 2-5 TF-IDF capped at 20,000; I is word 1-2 TF-IDF, stop words kept.
built = tf.build_blocks(train_texts, test_texts, CHOSEN)
X_full, X_test, feature_names = tf.stack(built, CHOSEN)

print(f"train {X_full.shape}, test {X_test.shape}")
for b in CHOSEN:
    print(f"  {b:<20} {built[b]['train'].shape[1]:>6,} columns")

## 3.3 The decision rule, and why it is not a 0.5 cut

This is where most of the project's score came from, so it is worth stating plainly.

The test file contains two populations distinguishable by id format: 1,999 rows with
UUID-style ids matching the training set's format, and 5,000 with numeric ids. They do
not share a class balance, and a single global threshold cannot serve both: corrections
that help one cancel the other, which is why the global share curve looked flat for so
long. Thresholding each group separately was worth +0.022.

We therefore set a target **share** per group, not a probability threshold. Share is the
fraction of rows labelled machine, and unlike a threshold it is comparable across models.
The shares below were located on the leaderboard, since no local split can inform them:
dev and holdout both inherit the training set's 62.52% balance, which is the one
distribution that does not apply to the test set.

## 3.4 The final model

In [ ]:
from lightgbm import LGBMClassifier

# Winner of the two-stage search in notebook 17, the first search run against the
# raw-text matrix rather than the supplied 5,000 columns. Selected on five length bands
# by paired per-fold difference against the defaults (better on 5 of 5 folds, paired
# +0.0135), then confirmed on the 3-band protocol it did not select on (+0.0046).
BEST_PARAMS = {
    "learning_rate": 0.04216444728733024,
    "n_estimators": 1093,
    "num_leaves": 130,
    "max_depth": 8,
    "min_child_samples": 63,
    "colsample_bytree": 0.37775424581021805,
    "subsample": 0.9180339679157208,
    "reg_alpha": 0.04680660143971163,
    "reg_lambda": 0.03734691045311837,
}
FIXED = dict(class_weight="balanced", subsample_freq=1, random_state=RANDOM_STATE,
             n_jobs=-1, verbose=-1)

model = LGBMClassifier(**BEST_PARAMS, **FIXED).fit(X_full, y_full)
scores = ensemble.member_score(model, X_test)
print(f"fitted on all {X_full.shape[0]:,} rows; test scores in "
      f"[{scores.min():.4f}, {scores.max():.4f}]")

In [ ]:
# Per-group shares, located on the leaderboard. uuid 0.6198 is close to the training
# prior of 0.6252, which is consistent with those rows sharing the training set's
# provenance; numeric sits well below it.
FINAL_SHARES = {"uuid": 0.6198, "numeric": 0.49}
HEDGE_SHARES = {"uuid": 0.6198, "numeric": 0.512}

groups = text.id_group(test_ids_txt)
masks = text.group_masks(test_ids_txt)

for fname, shares in [("tuned_lgbm_pergroup62_49.csv", FINAL_SHARES),
                      ("tuned_lgbm_pergroup62_51.csv", HEDGE_SHARES)]:
    # threshold_per_group takes an exact top-k per group rather than a quantile, so the
    # realised share matches the target even when scores tie.
    preds = clustering.threshold_per_group(scores, groups, shares)
    data.write_submission(test_ids_txt, preds, fname)

    sub = pd.read_csv(paths.SUBMISSIONS / fname, dtype={"id": str})
    assert list(sub["id"]) == list(sample["id"]), "id order drifted"
    assert set(sub["label"].unique()) <= {0, 1}
    lab = sub["label"].to_numpy()
    for g, m in masks.items():
        assert abs(lab[m].mean() - shares[g]) < 1 / m.sum(), f"{g} share off target"
    print(f"{fname}: uuid {lab[masks['uuid']].mean():.4f}, "
          f"numeric {lab[masks['numeric']].mean():.4f}, global {lab.mean():.4f}")

## 3.5 Results, and the two final submissions

| Submission | uuid share | numeric share | Kaggle public Macro F1 |
|---|---|---|---|
| **`tuned_lgbm_pergroup62_49.csv`** | 0.6198 | 0.4900 | **0.81249** |
| `tuned_lgbm_pergroup.csv` | 0.6198 | 0.4756 | 0.81224 |
| `tuned_lgbm_pergroup62_51.csv` | 0.6198 | 0.5120 | 0.80641 |

**The first two are a tie, not a ranking.** They are 0.00025 apart on files that differ
on 72 of 6,999 rows, which is 3% of the 0.0084 binomial noise floor on roughly 3,570
scored public rows. We do not claim the higher one is better.

**Our two final picks are `62_49` and `62_51`.** The second is deliberately not the
second-best public score. Every share coordinate in this project was fitted to the public
leaderboard, and a coordinate search of that size against a noise floor of 0.0084 can
manufacture 0.005 to 0.01 of gain that will not reappear on the private split. Picking
the top two public points would put both picks on one fitted coordinate; `62_51` is a
deliberate hedge one step away, and its 0.006 shortfall is the measured price of that
insurance.

**A 2x2 that separates the model from the decision rule.** Both factors were measured at
both levels of the other, which is what makes each margin interpretable:

| numeric share | LightGBM defaults | LightGBM tuned | tuned - defaults |
|---|---|---|---|
| 0.4756 | 0.80143 | 0.81224 | +0.01081 |
| 0.5120 | 0.80049 | 0.80641 | +0.00592 |

The tuned model wins at both share levels, mean +0.0084. Each cell alone sits at the
noise floor; it is the replication across two independent share points that makes the
result credible.

**Where the score came from.** From our first submission at 0.65738 to 0.81249, roughly
65% of the total gain came from how scores are turned into labels, 28% from what the
model sees, and 7% from the model itself. That ordering was not what we expected, and the
one result that revised it is the last row of the table above: the identical
hyperparameter search returned nothing on the supplied features and +0.0108 on the
raw-text representation. A null result belongs to the representation it was measured on
and does not travel to a different one.